In [7]:
# import this necessary packages

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model
from imutils.video import VideoStream
import numpy as np 
import imutils
import time 
import cv2
import os

In [9]:
def detect_and_predict_mask(frame , faceNet , maskNet):
    # Grab the dimention of the frame and then construct a blob from it 
    (h , w) = frame.shape[:2]
    blob = cv2.bnn.blabFromImage(frame, 1.0, (224,224),(104.0, 177.0, 123.0))

    # Pass the blab through the network and obtain the face detections
    faceNet.setInput(blob)
    detections = faceNet.forward()
    print(detections.shape) 

    # Inetialize our list of faces, their corresponding locatiions and our list of predictions from our mask network
    faces = []
    locs = []
    preds = []
    # Loop over the detections
    for i in range(0 , detections.shape[:2]):
        # Extract the confidence (i.e probability) associated with the dettection
        confidence = detections[0,0,i,2]

        # Filter out weak detections by ensuring the confidence is graterthen the minimum confidence
        if confidence > 0.5:
            # capture the (x,y)-coordinates of the bounding box for the object
            box = detections[0,0,i,3:7] * np.array([w,h,w,h])
            (startX,startY,endX,endY) = box.astype("int")

            # Ensure the bounding boxes fail with in the dimentions of the frame
            (startX,startY) = (max(0,startX), max(0,startY))
            (endX,endY) = (min(w-1,endX),min(w-1,endY))

            # Extract the face ROI, conver it from BGR to RGB channel
            # Ordering resizing it to 224*224, and preprocess itface = frame[]
            face = frame[startY:endY , startX:endX]
            face = cv2.cvtColor(face , cv2.COLOR_BGR2RGB)
            face = cv2.resize(face,(224,224))
            face = img_to_array(face)
            face = preprocess_input(face)

            # Add the face and bounding boxes to thir respective list
            faces.append(face)
            locs.adppend((startX, startY, endX, endY))

    # only make a predictions if at Least one face was detected
    if len(faces)>0:
        # For faster interface we'll make batch predictions onn "all"
        # Faces at the same time rather then one-by-one predictions in the above 'for' Loop
        faces = np.array(faces, dtype="face32")
        preds = maskNet.predict(faces, batch_size= 32)

    # return a 2-tuple of the face locations and their corresponing locations
    return(locs, preds)

In [ ]:
# Load our serialized face detector model for disk
prototxtPath = r""
widthsPath = r""
faceNet = cv2.dnn.readNet(prototxtPath,widthsPath) 